In [1]:
print("hello world")

hello world


In [2]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
import pickle

In [3]:
data = pd.read_csv("Churn_Modelling.csv")
data.head(5)

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
## Data preprocessing
### Remove the irrelevent colums from the table
data = data.drop(['RowNumber','CustomerId','Surname'], axis=1)
data.tail()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1
9999,792,France,Female,28,4,130142.79,1,1,0,38190.78,0


In [5]:
## Encode the categorical variables
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])
data.tail()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1
9999,792,France,0,28,4,130142.79,1,1,0,38190.78,0


In [6]:
## One hot encoding "Geography"
from sklearn.preprocessing import OneHotEncoder
onehot_encode_geo = OneHotEncoder()
geo_encoder = onehot_encode_geo.fit_transform(data[['Geography']])
geo_encoder

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [7]:
geo_encoder.toarray()

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [8]:
onehot_encode_geo.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [9]:
geo_encoder_df = pd.DataFrame(geo_encoder.toarray(),columns=onehot_encode_geo.get_feature_names_out(['Geography']))
geo_encoder_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [10]:
## Combine the one hot encoder with the original data
data = pd.concat([data.drop('Geography', axis=1), geo_encoder_df], axis=1)
data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


In [11]:
## Save the encodera and scaler
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encode_geo, file)

In [12]:
# Divide the dataset into dependent and independent
X = data.drop('EstimatedSalary', axis=1)
y = data['EstimatedSalary']

In [13]:
# Split the training and testing datasets
X_train, X_test, y_train , y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Scale thses features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [14]:
# save into pickle
with open('scaler_salary.pkl', 'wb') as file:
    pickle.dump(scaler, file)

### ANN Regression Problem statement

In [15]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import TensorBoard, EarlyStopping
import datetime

In [16]:
X_train.shape[1]

12

In [17]:
## Build our ANN Model
### In regression model at third Dense we dont need to use an activation funtion bcoz it default use linear activation funtion
### And also here in regressionwe use mean_absolute_error loss funtion or mean_squared_error
model = Sequential([
    Dense(64,activation = 'relu', input_shape = (X_train.shape[1],)), ## HL1 Connected with input layer
    Dense(32,activation = 'relu'), ## HL2
    Dense(1) ## Output layer for regression
])

### Compile the model

model.compile(optimizer='adam', loss="mean_absolute_error", metrics=['mae'])

In [18]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [19]:
## Set up the tensorflow
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

log_dir = "regressionlogs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir = log_dir, histogram_freq=1)

In [20]:
## Set up the early stopping
early_stopping_callbacks = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights= True)

In [21]:
## Train the model 
history = model.fit(
    X_train, y_train, validation_data = (X_test, y_test),
    epochs = 100,
    callbacks = [tensorflow_callback, early_stopping_callbacks]
)

Epoch 1/100


250/250 [==============================] - 4s 7ms/step - loss: 100378.0625 - mae: 100378.0625 - val_loss: 98514.2344 - val_mae: 98514.2344
Epoch 2/100
250/250 [==============================] - 1s 5ms/step - loss: 99609.8750 - mae: 99609.8750 - val_loss: 96957.8594 - val_mae: 96957.8594
Epoch 3/100
250/250 [==============================] - 1s 5ms/step - loss: 96892.0547 - mae: 96892.0547 - val_loss: 92961.7578 - val_mae: 92961.7578
Epoch 4/100
250/250 [==============================] - 1s 5ms/step - loss: 91563.6719 - mae: 91563.6719 - val_loss: 86353.0859 - val_mae: 86353.0859
Epoch 5/100
250/250 [==============================] - 1s 5ms/step - loss: 83800.4141 - mae: 83800.4141 - val_loss: 77763.7891 - val_mae: 77763.7891
Epoch 6/100
250/250 [==============================] - 1s 5ms/step - loss: 74672.3516 - mae: 74672.3516 - val_loss: 68904.0547 - val_mae: 68904.0547
Epoch 7/100
250/250 [==============================] - 1s 5ms/step - loss: 65842.8516 - mae: 65842.851

In [22]:
## Save the model
model.save('regressionmodel.h5')

c:\Users\KIRUBAK\Desktop\Gen AI\.venv\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [23]:
## Load tensorboard Extension
%load_ext tensorboard

In [25]:
%tensorboard --logdir regressionlogs/fit

Reusing TensorBoard on port 6008 (pid 36788), started 5:13:48 ago. (Use '!kill 36788' to kill it.)

In [26]:
## Evaluate the model
test_loss, test_mae = model.evaluate(X_test, y_test)
print(f'Test Mae: {test_mae}')

63/63 [==============================] - 0s 3ms/step - loss: 50320.5273 - mae: 50320.5273
Test Mae: 50320.52734375
